# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f\nAuthors: {dataset.metadata.author}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Keywords: {dataset.metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their IDs as described in the Croissant schema.

In [ ]:
# List available record sets, their @id, and their fields' @id.
all_record_sets = dataset.metadata.recordSet  # This is a list of record set objects

print("Available Record Sets and their Fields (@id):\n")
record_set_ids = []
for rset in all_record_sets:
    rset_id = rset['@id'] if isinstance(rset, dict) else rset._id
    name = rset['name'] if isinstance(rset, dict) and 'name' in rset else getattr(rset, 'name', None)
    print(f"- {rset_id}  ({name})")
    if hasattr(rset, 'field'):
        print("  Fields (by @id):")
        for field in rset.field:
            # field may be a dict or Field obj
            field_id = field['@id'] if isinstance(field, dict) else field._id
            field_name = field['name'] if isinstance(field, dict) and 'name' in field else getattr(field, 'name', None)
            print(f"    - {field_id}  ({field_name})")
    print("")
    record_set_ids.append(rset_id)
num_record_sets = len(all_record_sets)
if num_record_sets == 0:
    print('WARNING: No record sets are defined in the Croissant schema, or their enumeration is empty.')

## 3. Data Extraction
Load data from record sets into Pandas DataFrames.

> Note: We use each record set's `@id` as required. If the dataset contains multiple record sets, this cell will aggregate them in one dictionary keyed by their `@id`.

If no record sets are enumerated, you may need to check the documentation for access mechanisms or refer to individual file objects in the schema.

In [ ]:
# Form the list of record set @id's
record_sets = record_set_ids  # Populated from the previous cell
dataframes = dict()

# Try to extract a few records for each record set
for record_set_id in record_sets:
    print(f"Extracting records for {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"{record_set_id} columns: {df.columns.tolist()}")
        display(df.head())
    except Exception as ex:
        print(f"Could not extract records from {record_set_id} due to: {ex}\n")

if not dataframes:
    print("No data loaded. Please check available record sets and file objects.")

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps, such as filtering records and normalizing numeric fields. All field/column references use their `@id`.

**Example:**
- Filter for proteins with coverage greater than a threshold.
- Normalize a numeric field.
- Group by a categorical field (e.g., sample or protein class).

If no data was loaded above, this cell will be skipped.

In [ ]:
# Pick the first record set for analysis
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Analyzing record set: {record_set_id}\nColumns: {df.columns.tolist()}")

    # Try to auto-detect a numeric field; else, specify by @id manually
    numeric_field_candidates = [c for c in df.columns if np.issubdtype(df[c].dtype, np.number)]
    if not numeric_field_candidates:
        # Try casting columns to numeric if possible
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c])
            except Exception:
                continue
        numeric_field_candidates = [c for c in df.columns if np.issubdtype(df[c].dtype, np.number)]

    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]  # e.g., '@id' of 'coverage_percentage' field
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):\n", filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try to find a group field
        group_field_candidates = [c for c in df.columns if c != numeric_field_id]
        group_field = group_field_candidates[0] if group_field_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric field detected in this record set for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields using Matplotlib/Seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    # Plot distribution for the normalized numeric field
    if 'filtered_df' in locals() and not filtered_df.empty:
        plt.figure(figsize=(8,4))
        field = f"{numeric_field_id}_normalized"
        sns.histplot(filtered_df[field], bins=20)
        plt.title(f"Distribution of normalized {numeric_field_id}")
        plt.xlabel(field)
        plt.ylabel("Count")
        plt.show()
        # Optionally, scatter plot by group field if available
        if group_field:
            plt.figure(figsize=(8,5))
            sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
            plt.title(f"{numeric_field_id} by {group_field}")
            plt.xticks(rotation=45)
            plt.show()
    else:
        print("Not enough data for visualization.")
else:
    print("No data to visualize.")

## 6. Conclusion
This notebook demonstrated loading and exploring the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

- The data includes extensive protein records and experimental metadata, referenced by their Croissant `@id`.
- With `mlcroissant`, record sets and field relationships can be programmatically navigated for scalable processing.
- Exploratory analysis illustrated simple filtering, normalization, and basic visualization workflows.

Consider using advanced feature selection, domain annotation, and integrating additional sample metadata for comprehensive proteomics or biomedical analyses.